# Executive Vibe Check — Track A Pipeline
**WUSS Academic Journal Committee · Spring 2026**

This notebook computes NLP-derived scores for every earnings call transcript in the Strux dataset. The output is a CSV file used as the primary data asset for the project.

**Output files:**
- `strux_scores.csv` — sentiment and uncertainty scores for all transcripts
- `strux_manifest.csv` — ticker and date pairs for Track B (volatility computation)

## 1. Setup and Imports

In [ ]:
from datasets import load_dataset
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import torch.nn.functional as F

## 2. Load the Strux Dataset

Strux contains 11,411 quarterly earnings call transcripts from S&P 500 and NASDAQ 500 companies covering 2017–2024. It is hosted on Hugging Face and was originally constructed for an investment decision classification task.

We combine the train and test splits into a single flat pool. The split was designed for their classifier and is irrelevant to our regression analysis — we want all transcripts.

Each transcript has the following fields:
- `ticker` — stock ticker
- `date` — earnings call date (YYYY-MM-DD)
- `prepared_remarks` — list of dicts with `name` and `speech` keys
- `questions_and_answers` — same structure, Q&A portion of the call

In [ ]:
print("Loading Strux dataset...")
transcripts = load_dataset("BUILDERlym/STRUX-Transcripts")
all_data = list(transcripts['full'])
print(f"Total transcripts: {len(all_data)}")

# Inspect one transcript to confirm schema
sample = all_data[0]
print(f"\nSample ticker: {sample['ticker']}")
print(f"Sample date: {sample['date']}")
print(f"Prepared remarks (first entry): {sample['prepared_remarks'][0]}")
print(f"Q&A (first entry): {sample['questions_and_answers'][0]}")

In [ ]:
full_data = list(transcripts['full'])
sample_dates = [(t['ticker'], t['date']) for t in full_data[:5]]
print(sample_dates)

train_amzn = [(t['ticker'], t['date']) for t in transcripts['train']
              if t['ticker'] == 'AMZN'][:3]
print(train_amzn)

# Check if AMZN 2023-02-02 is in full
in_full = any(t['ticker'] == 'AMZN' and t['date'] == '2023-02-02'
              for t in full_data)
print(f"AMZN 2023-02-02 in full split: {in_full}")

## 3. Load FinBERT

FinBERT is a BERT model fine-tuned on financial text. Source: `ProsusAI/finbert` on Hugging Face.

It outputs three class probabilities: positive, negative, neutral. We convert these to a continuous sentiment score:

**sentiment_score = P(positive) − P(negative)**

This gives a value in [−1, 1] where positive values indicate optimistic tone and negative values indicate pessimistic tone. The label order is `{0: positive, 1: negative, 2: neutral}` — confirmed below.

We move the model to GPU if available (strongly recommended — CPU inference on ~12,000 transcripts is infeasibly slow).

In [ ]:
print("Loading FinBERT...")
tokenizer = BertTokenizer.from_pretrained("ProsusAI/finbert")
model = BertForSequenceClassification.from_pretrained("ProsusAI/finbert")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print(f"Device: {device}")
print(f"Label order: {model.config.id2label}")

## 4. Text Extraction

The Strux schema stores remarks as a list of dicts:
`[{'name': 'Speaker Name', 'speech': ['segment1', 'segment2']}, ...]`

We extract all speech segments from the list regardless of speaker. This includes the Operator and IR representative in addition to the CEO and CFO. Their segments are short and low-information relative to the full transcript, so the effect on the aggregate score is minimal. Filtering by role is a noted future refinement.

We apply the same extraction logic to both `prepared_remarks` and `questions_and_answers`.

**Why prepared remarks vs. Q&A?**
The standard in the textual analysis literature (Loughran & McDonald 2011) is to use scripted management speech. Prepared remarks are pre-written and reflect deliberate communication choices. Q&A is unscripted and may reveal different information — Poyraz Ozer raised this point citing recent literature suggesting Q&A has significant information content. We compute scores for both and include Q&A as supplementary columns so the regression can test whether Q&A adds predictive power beyond prepared remarks.

In [ ]:
def extract_text(remarks: list) -> list:
    segments = []
    for r in remarks:
        if isinstance(r, str):
            segments.append(r)
        elif isinstance(r, dict):
            segments.extend(r.get('speech', []))
    return [s for s in segments if s.strip()]

# Q&A uses identical extraction logic
extract_qa_text = extract_text

## 5. FinBERT Sentiment Scoring

FinBERT has a 512-token input limit. Earnings call transcripts are much longer, so we split into segments and score each individually using batched inference, then take the mean.

**Aggregation decision:** Simple mean across segments, treating each segment equally. An alternative is weighting by segment length — noted as a robustness check for the paper.

**Batching:** We pass segments through the model in batches of 16 rather than one at a time. This uses GPU parallelism and reduces runtime from hours to minutes.

In [ ]:
def aggregate_sentiment_batched(segments: list, batch_size: int = 16) -> float:
    if not segments:
        return 0.0
    all_scores = []
    for i in range(0, len(segments), batch_size):
        batch = segments[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", truncation=True,
            max_length=512, padding=True
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
        scores = (probs[:, 0] - probs[:, 1]).tolist()  # P(pos) - P(neg)
        all_scores.extend(scores)
    return sum(all_scores) / len(all_scores)

## 6. Loughran-McDonald Uncertainty Scoring

The uncertainty score measures the frequency of hedging language in the transcript.

**Formula:** count of LM uncertainty words / total word count

The Loughran-McDonald Master Dictionary (Notre Dame SRAF) flags words associated with vagueness and hedging — e.g. "approximately", "could", "might", "uncertain". Normalizing by total word count allows comparison across transcripts of different lengths.

This is distinct from sentiment — a transcript can be positive in tone but high in uncertainty (e.g. optimistic but hedged). H2 of our research hypothesizes that uncertainty score has greater predictive power over post-call volatility than sentiment score alone.

In [ ]:
lm_url = "https://raw.githubusercontent.com/darrentweng/wharton-uss-academic-journal-executive-sentiment/refs/heads/main/data/LM_MasterDictionary.csv"
lm = pd.read_csv(lm_url)
lm_uncertainty = set(lm[lm['Uncertainty'] != 0]['Word'].str.upper())
print(f"LM uncertainty words loaded: {len(lm_uncertainty)}")

def get_uncertainty_score(segments: list) -> float:
    full_text = " ".join(segments).upper()
    words = full_text.split()
    if not words:
        return 0.0
    return sum(1 for w in words if w in lm_uncertainty) / len(words)

## 7. Sanity Check

Before running the full pipeline, verify the functions work correctly on one transcript and the output values look sensible.

Expected ranges:
- `sentiment_score`: most earnings calls are moderately positive (0.1–0.5)
- `uncertainty_score`: small positive fraction, typically 0.01–0.05
- Q&A scores may differ meaningfully from prepared remarks scores

In [ ]:
sample = all_data[0]
remarks_segments = extract_text(sample['prepared_remarks'])
qa_segments = extract_qa_text(sample['questions_and_answers'])

print(f"Ticker: {sample['ticker']}  |  Date: {sample['date']}")
print(f"Prepared remarks segments: {len(remarks_segments)}")
print(f"Q&A segments: {len(qa_segments)}")
print(f"\nsentiment_score:      {aggregate_sentiment_batched(remarks_segments):.6f}")
print(f"uncertainty_score:    {get_uncertainty_score(remarks_segments):.6f}")
print(f"sentiment_score_qa:   {aggregate_sentiment_batched(qa_segments):.6f}")
print(f"uncertainty_score_qa: {get_uncertainty_score(qa_segments):.6f}")

## 8. Full Pipeline

Process all 11,411 transcripts. Checkpoints are saved every 500 transcripts in case the Colab session times out.

Estimated runtime: ~15–30 minutes on a T4 GPU. To enable: **Runtime → Change runtime type → T4 GPU**

In [ ]:
import os

if os.path.exists("strux_scores_checkpoint.csv"):
    existing = pd.read_csv("strux_scores_checkpoint.csv")
    results = existing.to_dict('records')
    start_idx = len(results)
    print(f"Resuming from index {start_idx}")
else:
    results = []
    start_idx = 0

for i, transcript in enumerate(all_data):
    if i % 100 == 0:
        print(f"Processing {i}/{len(all_data)}...")
    if i % 500 == 0 and i > 0:
        pd.DataFrame(results).to_csv("strux_scores_checkpoint.csv", index=False)
        print(f"  Checkpoint saved at {i}")

    remarks = transcript['prepared_remarks']
    qa = transcript['questions_and_answers']

    if not remarks:
        continue

    remarks_segments = extract_text(remarks)
    qa_segments = extract_qa_text(qa) if qa else []

    results.append({
        "ticker":               transcript['ticker'].upper(),
        "date":                 transcript['date'],
        "sentiment_score":      round(aggregate_sentiment_batched(remarks_segments), 6),
        "uncertainty_score":    round(get_uncertainty_score(remarks_segments), 6),
        "sentiment_score_qa":   round(aggregate_sentiment_batched(qa_segments), 6) if qa_segments else None,
        "uncertainty_score_qa": round(get_uncertainty_score(qa_segments), 6) if qa_segments else None,
    })

print("Done.")

## 9. Output

In [ ]:
df = pd.DataFrame(results)
df.to_csv("strux_scores.csv", index=False)
print(f"strux_scores.csv: {len(df)} rows")
print(df[['sentiment_score', 'uncertainty_score',
          'sentiment_score_qa', 'uncertainty_score_qa']].describe())

manifest = df[["ticker", "date"]]
manifest.to_csv("strux_manifest.csv", index=False)
print(f"\nstrux_manifest.csv: {len(manifest)} rows")